In [ ]:
# Install required packages (runs automatically in Colab, fast no-op in Binder)
!pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Quantum teleportation

Quantum teleportation, or just teleportation for short, is a protocol where a sender (Alice) transmits a qubit to a receiver (Bob) by making use of a shared entangled quantum state (one e-bit, to be specific) along with two bits of classical communication.
The name *teleportation* is meant to be suggestive of the concept in science fiction where matter is transported from one location to another by a futuristic process, but it must be understood that matter is not teleported in quantum teleportation — what is actually teleported is quantum information.

The set-up for teleportation is as follows.

We assume that Alice and Bob share an e-bit: Alice holds a qubit $\mathsf{A},$ Bob holds a qubit $\mathsf{B},$ and together the pair $(\mathsf{A},\mathsf{B})$ is in the state $\vert\phi^+\rangle.$
It could be, for instance, that Alice and Bob were in the same location in the past, they prepared the qubits $\mathsf{A}$ and $\mathsf{B}$ in the state $\vert \phi^+ \rangle,$ and then each went their own way with their qubit in hand.
Or, it could be that a different process, such as one involving a third party or a complex distributed process, was used to establish this shared e-bit.
These details are not part of the teleportation protocol itself.

Alice then comes into possession of a third qubit $\mathsf{Q}$ that she wishes to transmit to Bob.
The state of the qubit $\mathsf{Q}$ is considered to be *unknown* to Alice and Bob, and no assumptions are made about it.
For example, the qubit $\mathsf{Q}$ might be entangled with one or more other systems that neither Alice nor Bob can access.
To say that Alice wishes to transmit the qubit $\mathsf{Q}$ to Bob means that Alice would like Bob to be holding a qubit that is in the same state that $\mathsf{Q}$ was in at the start of the protocol, having whatever correlations that $\mathsf{Q}$ had with other systems, as if Alice had physically handed $\mathsf{Q}$ to Bob.

We could imagine that Alice physically sends the qubit $\mathsf{Q}$ to Bob, and if it reaches Bob without being altered or disturbed in transit, then Alice and Bob's task will be accomplished.
In the context of teleportation, however, it is our assumption that this is not feasible; Alice cannot send qubits directly to Bob.
She may, however, send classical information to Bob.

These are reasonable assumptions in a variety of settings.
For example, if Alice doesn't know Bob's exact location, or the distance between them is large, physically sending a qubit using the technology of today, or the foreseeable future, would be challenging to say the least.
However, as we know from everyday experiences, classical information transmission under these circumstances is quite straightforward.

At this point, one might ask whether it is possible for Alice and Bob to accomplish their task without even needing to make use of a shared e-bit.
In other words, is there any way to transmit a qubit using classical communication alone?

The answer is no, it is not possible to transmit quantum information using classical communication alone.
This is not too difficult to prove mathematically using basic quantum information theory, but we can alternatively rule out the possibility of transmitting qubits using classical communication alone by thinking about the no-cloning theorem.

Imagine that there was a way to send quantum information using classical communication alone.
Classical information can easily be copied and broadcast, which means that any classical transmission from Alice to Bob might also be received by a second receiver (Charlie, let us say).
But if Charlie receives the same classical communication that Bob received, then would he not also be able to obtain a copy of the qubit $\mathsf{Q}?$
This would suggest that $\mathsf{Q}$ was cloned, which we already know is impossible by the no-cloning theorem, and so we conclude that there is no way to send quantum information using classical communication alone.

When the assumption that Alice and Bob share an e-bit is in place, however, it is possible for Alice and Bob to accomplish their task.
This is precisely what the quantum teleportation protocol does.

## Protocol

Here is a quantum circuit diagram that describes the teleportation protocol:

![Teleportation circuit](../../../../learning/images/courses/basics-of-quantum-information/entanglement-in-action/teleportation.svg)

The diagram is slightly stylized in that it depicts the separation between Alice and Bob, with two diagonal wires representing classical bits that are sent from Alice to Bob, but otherwise it is an ordinary quantum circuit diagram.
The qubit names are shown above the wires rather than to the left so that the initial states can be shown as well (which we will commonly do when it is convenient).
It should also be noted that the $X$ and $Z$ gates have *classical* controls, which simply means that the gates are not applied or applied depending on whether these classical control bits are $0$ or $1,$ respectively.

In words, the teleportation protocol is as follows:

1. Alice performs a controlled-NOT operation on the pair $(\mathsf{A},\mathsf{Q}),$ with $\mathsf{Q}$ being the control and $\mathsf{A}$ being the target, and then performs a Hadamard operation on $\mathsf{Q}.$

2. Alice then measures both $\mathsf{A}$ and $\mathsf{Q},$ with respect to a standard basis measurement in both cases, and transmits the classical outcomes to Bob. Let us refer to the outcome of the measurement of $\mathsf{A}$ as $a$ and the outcome of the measurement of $\mathsf{Q}$ as $b.$

3. Bob receives $a$ and $b$ from Alice, and depending on the values of these bits he performs these operations:
   - If $a = 1,$ then Bob performs a bit flip (or $X$ gate) on his qubit $\mathsf{B}.$
   - If $b = 1,$ then Bob performs a phase flip (or $Z$ gate) on his qubit $\mathsf{B}.$

   That is, conditioned on $ab$ being $00,$ $01,$ $10,$ or $11,$ Bob performs one of the operations $\mathbb{I},$ $Z,$ $X,$ or $ZX$ on the qubit $\mathsf{B}.$

This is the complete description of the teleportation protocol.
The analysis that appears below reveals that when it is run, the qubit $\mathsf{B}$ will be in whatever state $\mathsf{Q}$ was in prior to the protocol being executed, including whatever correlations it had with any other systems — which is to say that the protocol has effectively implemented a perfect qubit communication channel, where the state of $\mathsf{Q}$ has been "teleported" into $\mathsf{B}.$

Before proceeding to the analysis, notice that this protocol does not succeed in cloning the state of $\mathsf{Q},$ which we already know is impossible by the no-cloning theorem.
Rather, when the protocol is finished, the state of the qubit $\mathsf{Q}$ will have changed from its original value to $\vert b\rangle$ as a result of the measurement performed on it.
Also notice that the e-bit has effectively been "burned" in the process: the state of $\mathsf{A}$ has changed to $\vert a\rangle$ and is no longer entangled with $\mathsf{B}$ (or any other system).
This is the cost of teleportation.

## Analysis

To analyze the teleportation protocol, we'll examine the behavior of the circuit described above, one step at a time, beginning with the situation in which $\mathsf{Q}$ is initially in the state $\alpha\vert 0\rangle + \beta\vert 1\rangle.$
This is not the most general situation, as it does not capture the possibility that $\mathsf{Q}$ is entangled with other systems, but starting with this simpler case will add clarity to the analysis.
The more general case is addressed below, following the analysis of the simpler case.

Specifically, we will consider the states of the qubits $(\mathsf{B},\mathsf{A},\mathsf{Q})$ at the times suggested by this figure:

![Teleportation circuit time-steps](../../../../learning/images/courses/basics-of-quantum-information/entanglement-in-action/teleportation-time-steps.svg)

Under the assumption that the qubit $\mathsf{Q}$ begins the protocol in the state $\alpha\vert 0\rangle + \beta\vert 1\rangle,$ the state of the three qubits $(\mathsf{B},\mathsf{A},\mathsf{Q})$ together at the start of the protocol is therefore

$$
\vert \pi_0 \rangle
= \vert \phi^+\rangle \otimes
\bigl(\alpha\vert 0\rangle + \beta\vert 1\rangle \bigr)
= \frac{\alpha \vert 000 \rangle + \alpha \vert 110\rangle + \beta \vert 001\rangle + \beta \vert 111\rangle}{\sqrt{2}}.
$$

The first gate that is performed is the controlled-NOT gate, which transforms the state $\vert\pi_0\rangle$ into

$$
\vert \pi_1 \rangle  = \frac{\alpha \vert 000 \rangle + \alpha \vert 110\rangle + \beta \vert 011\rangle + \beta \vert 101\rangle}{\sqrt{2}}.
$$

Then the Hadamard gate is applied, which transforms the state $\vert\pi_1\rangle$ into

$$
\begin{aligned}
\vert\pi_2\rangle
& =
\frac{\alpha \vert 00\rangle \vert + \rangle + \alpha \vert 11\rangle\vert +\rangle + \beta \vert 01\rangle\vert -\rangle + \beta \vert 10\rangle\vert -\rangle}{\sqrt{2}}\\[2mm]
& = \frac{\alpha \vert 000 \rangle
+ \alpha \vert 001 \rangle
+ \alpha \vert 110 \rangle
+ \alpha \vert 111 \rangle
+ \beta \vert 010 \rangle
- \beta \vert 011 \rangle
+ \beta \vert 100 \rangle
- \beta \vert 101 \rangle}{2}.
\end{aligned}
$$

Using the multilinearity of the tensor product, we may alternatively write this state as follows:

$$
\begin{aligned}
\vert\pi_2\rangle = \quad
& \frac{1}{2} \bigl(\alpha\vert 0 \rangle + \beta \vert 1\rangle \bigr)\vert 00\rangle \\[2mm]
+ & \frac{1}{2} \bigl(\alpha\vert 0 \rangle - \beta \vert 1\rangle \bigr)\vert 01\rangle \\[2mm]
+ & \frac{1}{2} \bigl(\alpha\vert 1 \rangle + \beta \vert 0\rangle \bigr)\vert 10\rangle \\[2mm]
+ & \frac{1}{2} \bigl(\alpha\vert 1 \rangle - \beta \vert 0\rangle \bigr)\vert 11\rangle.
\end{aligned}
$$

At first glance, it might look like something magical has happened, because the leftmost qubit $\mathsf{B}$ now seems to depend on the numbers $\alpha$ and $\beta,$ even though there has not yet been any communication from Alice to Bob.
This is an illusion.
Scalars float freely through tensor products, so $\alpha$ and $\beta$ are neither more nor less associated with the leftmost qubit than they are with the other qubits, and all we have done is to use algebra to express the state in a way that facilitates an analysis of the measurements.

Now let us consider the four possible outcomes of Alice's standard basis measurements, together with the actions that Bob performs as a result.

### Possible outcomes

 - The outcome of Alice's measurement is $aq = 00$ with probability

   $$
   \Biggl\| \frac{1}{2}\bigl(\alpha \vert 0\rangle + \beta\vert 1\rangle\bigr) \Biggr\|^2
   = \frac{\vert\alpha\vert^2 + \vert\beta\vert^2}{4} = \frac{1}{4},
   $$

   in which case the state of $(\mathsf{B},\mathsf{A},\mathsf{Q})$ becomes

   $$
   \bigl( \alpha \vert 0 \rangle + \beta \vert 1 \rangle \bigr) \vert 00 \rangle.
   $$

   Bob does nothing in this case, and so this is the final state of these three qubits.

 - The outcome of Alice's measurement is $aq = 01$ with probability

   $$
   \Biggl\| \frac{1}{2}\bigl(\alpha \vert 0\rangle - \beta\vert 1\rangle\bigr) \Biggr\|^2
   = \frac{\vert\alpha\vert^2 + \vert{-\beta}\vert^2}{4} = \frac{1}{4},
   $$

   in which case the state of $(\mathsf{B},\mathsf{A},\mathsf{Q})$ becomes

   $$
   \bigl( \alpha \vert 0 \rangle - \beta \vert 1 \rangle \bigr) \vert 01 \rangle.
   $$

   In this case Bob applies a $Z$ gate to $\mathsf{B},$ leaving $(\mathsf{B},\mathsf{A},\mathsf{Q})$ in the state

   $$
   \bigl( \alpha \vert 0 \rangle + \beta \vert 1 \rangle \bigr) \vert 01 \rangle.
   $$

 - The outcome of Alice's measurement is $aq = 10$ with probability

   $$
   \Biggl\| \frac{1}{2}\bigl(\alpha \vert 1\rangle + \beta\vert 0\rangle\bigr) \Biggr\|^2
   = \frac{\vert\alpha\vert^2 + \vert\beta\vert^2}{4} = \frac{1}{4},
   $$

   in which case the state of $(\mathsf{B},\mathsf{A},\mathsf{Q})$ becomes

   $$
   \bigl( \alpha \vert 1 \rangle + \beta \vert 0 \rangle \bigr) \vert 10 \rangle.
   $$

   In this case, Bob applies an $X$ gate to the qubit $\mathsf{B},$ leaving $(\mathsf{B},\mathsf{A},\mathsf{Q})$ in the state

   $$
   \bigl( \alpha \vert 0 \rangle + \beta \vert 1 \rangle \bigr) \vert 10 \rangle.
   $$

 - The outcome of Alice's measurement is $aq = 11$ with probability

   $$
   \Biggl\| \frac{1}{2}\bigl(\alpha \vert 1\rangle - \beta\vert 0\rangle\bigr) \Biggr\|^2
   = \frac{\vert\alpha\vert^2 + \vert{-\beta}\vert^2}{4} = \frac{1}{4},
   $$

   in which case the state of $(\mathsf{B},\mathsf{A},\mathsf{Q})$ becomes

   $$
   \bigl( \alpha \vert 1 \rangle - \beta \vert 0 \rangle \bigr) \vert 11 \rangle.
   $$

   In this case, Bob performs the operation $ZX$ on the qubit $\mathsf{B},$ leaving $(\mathsf{B},\mathsf{A},\mathsf{Q})$ in the state

   $$
   \bigl( \alpha \vert 0 \rangle + \beta \vert 1 \rangle \bigr) \vert 11 \rangle.
   $$

We now see, in all four cases, that Bob's qubit $\mathsf{B}$ is left in the state
$\alpha\vert 0\rangle + \beta\vert 1\rangle$ at the end of the protocol, which is the initial state of the qubit $\mathsf{Q}.$
This is what we wanted to show: the teleportation protocol has worked correctly.

We also see that the qubits $\mathsf{A}$ and $\mathsf{Q}$ are left in one of the four states $\vert 00\rangle,$ $\vert 01\rangle,$ $\vert 10\rangle,$ or $\vert 11\rangle,$ each with probability $1/4,$ depending upon the measurement outcomes that Alice obtained.
Thus, as was already suggested above, at the end of the protocol Alice no longer has the state $\alpha \vert 0\rangle + \beta \vert 1\rangle,$ which is consistent with the no-cloning theorem.

Notice that Alice's measurements yield absolutely no information about the state $\alpha \vert 0\rangle + \beta \vert 1\rangle.$
That is, the probability for each of the four possible measurement outcomes is $1/4,$ irrespective of $\alpha$ and $\beta.$
This is also essential for teleportation to work correctly.
Extracting information from an unknown quantum state necessarily disturbs it in general, but here Bob obtains the state without it being disturbed.

Now let's consider the more general situation in which the qubit $\mathsf{Q}$ is initially entangled with another system, which we'll name $\mathsf{R}.$
A similar analysis to the one above reveals that the teleportation protocol functions correctly in this more general case:
at the end of the protocol, the qubit $\mathsf{B}$ held by Bob is entangled with $\mathsf{R}$ in the same way that $\mathsf{Q}$ was at the start of the protocol, as if Alice had simply handed $\mathsf{Q}$ to Bob.

To prove this, let us suppose that the state of the pair $(\mathsf{Q},\mathsf{R})$ is initially given by a quantum state vector of the form

$$
\alpha \vert 0 \rangle_{\mathsf{Q}} \vert \gamma_0\rangle_{\mathsf{R}}
+ \beta \vert 1 \rangle_{\mathsf{Q}} \vert \gamma_1\rangle_{\mathsf{R}},
$$

where $\vert\gamma_0\rangle$ and $\vert\gamma_1\rangle$ are quantum state vectors for the system $\mathsf{R}$ and $\alpha$ and $\beta$ are complex numbers satisfying $\vert \alpha \vert^2 + \vert\beta\vert^2 = 1.$
Any quantum state vector of the pair $(\mathsf{Q},\mathsf{R})$ can be expressed in this way.

The following figure depicts the same circuit as before, with the addition of the system $\mathsf{R}$ (represented by a collection of qubits on the top of the diagram that nothing happens to).

![Teleportation with an entangled input](../../../../learning/images/courses/basics-of-quantum-information/entanglement-in-action/teleportation-with-entanglement.svg)

To analyze what happens when the teleportation protocol is run, it is helpful to permute the systems, along the same lines that was described in the previous lesson.
Specifically, we'll consider the state of the systems in the order $(\mathsf{B},\mathsf{R},\mathsf{A},\mathsf{Q})$ rather than $(\mathsf{B},\mathsf{A},\mathsf{Q},\mathsf{R}).$
The names of the various systems are included as subscripts in the expressions that follow for clarity.

At the start of the protocol, the state of these systems is as follows:

$$
\begin{aligned}
\vert \pi_0\rangle
& = \vert \phi^+\rangle_{\mathsf{BA}} \otimes \bigl(
  \alpha \vert 0\rangle_{\mathsf{Q}} \vert\gamma_0\rangle_{\mathsf{R}}
+ \beta \vert 1\rangle_{\mathsf{Q}}\vert\gamma_1\rangle_{\mathsf{R}}\bigr)\\[1mm]
& = \frac{
  \alpha \vert 0\rangle_{\mathsf{B}} \vert \gamma_0 \rangle_{\mathsf{R}} \vert 00 \rangle_{\mathsf{AQ}}
+ \alpha \vert 1\rangle_{\mathsf{B}} \vert \gamma_0 \rangle_{\mathsf{R}} \vert 10 \rangle_{\mathsf{AQ}}
+ \beta \vert 0\rangle_{\mathsf{B}} \vert \gamma_1 \rangle_{\mathsf{R}} \vert 01 \rangle_{\mathsf{AQ}}
+ \beta \vert 1\rangle_{\mathsf{B}} \vert \gamma_1 \rangle_{\mathsf{R}} \vert 11 \rangle_{\mathsf{AQ}}}{\sqrt{2}}.
\end{aligned}
$$

First the controlled-NOT gate is applied, which transforms this state to

$$
\vert\pi_1\rangle =
\frac{
  \alpha \vert 0\rangle_{\mathsf{B}} \vert\gamma_0 \rangle_{\mathsf{R}} \vert 00\rangle_{\mathsf{AQ}}
+ \alpha \vert 1\rangle_{\mathsf{B}} \vert\gamma_0 \rangle_{\mathsf{R}} \vert 10\rangle_{\mathsf{AQ}}
+ \beta \vert 0\rangle_{\mathsf{B}} \vert\gamma_1 \rangle_{\mathsf{R}} \vert 11\rangle_{\mathsf{AQ}}
+ \beta \vert 1\rangle_{\mathsf{B}} \vert\gamma_1 \rangle_{\mathsf{R}} \vert 01\rangle_{\mathsf{AQ}}}{\sqrt{2}}.
$$

Then the Hadamard gate is applied.
After expanding and simplifying the resulting state, along similar lines to the analysis of the simpler case above, we obtain this expression of the resulting state:

$$
\begin{aligned}
\vert \pi_2 \rangle = \quad
  & \frac{1}{2} \bigl(
   \alpha \vert 0\rangle_{\mathsf{B}} \vert\gamma_0\rangle_{\mathsf{R}}
  + \beta \vert 1\rangle_{\mathsf{B}} \vert\gamma_1\rangle_{\mathsf{R}}
  \bigr) \vert 00\rangle_{\mathsf{AQ}}\\[2mm]
  + & \frac{1}{2} \bigl(
   \alpha \vert 0\rangle_{\mathsf{B}} \vert\gamma_0\rangle_{\mathsf{R}}
  - \beta \vert 1\rangle_{\mathsf{B}} \vert\gamma_1\rangle_{\mathsf{R}}
  \bigr) \vert 01\rangle_{\mathsf{AQ}}\\[2mm]
  + & \frac{1}{2} \bigl(
   \alpha \vert 1\rangle_{\mathsf{B}} \vert\gamma_0\rangle_{\mathsf{R}}
  + \beta \vert 0\rangle_{\mathsf{B}} \vert\gamma_1\rangle_{\mathsf{R}}
  \bigr) \vert 10\rangle_{\mathsf{AQ}}\\[2mm]
  + & \frac{1}{2} \bigl(
   \alpha \vert 1\rangle_{\mathsf{B}} \vert\gamma_0\rangle_{\mathsf{R}}
  - \beta \vert 0\rangle_{\mathsf{B}} \vert\gamma_1\rangle_{\mathsf{R}}
  \bigr) \vert 11\rangle_{\mathsf{AQ}}.
\end{aligned}
$$

Proceeding exactly as before, where we consider the four different possible outcomes of Alice's measurements along with the corresponding actions performed by Bob, we find that at the end of the protocol, the state of $(\mathsf{B},\mathsf{R})$ is always

$$
\alpha \vert 0 \rangle \vert \gamma_0\rangle + \beta \vert 1 \rangle \vert \gamma_1\rangle.
$$

Informally speaking, the analysis does not change in a significant way as compared with the simpler case above;
$\vert\gamma_0\rangle$ and $\vert\gamma_1\rangle$ essentially just "come along for the ride."
So, teleportation succeeds in creating a perfect quantum communication channel, effectively transmitting the contents of the qubit $\mathsf{Q}$ into $\mathsf{B}$ and preserving all correlations with other systems.

This is actually not surprising at all, given the analysis of the simpler case above.
As that analysis revealed, we have a physical process that acts like the identity operation on a qubit in an arbitrary quantum state, and there's only one way that can happen: the operation implemented by the protocol must *be* the identity operation.
That is, once we know that teleportation works correctly for a single qubit in isolation, we can conclude that the protocol effectively implements a perfect, noiseless quantum channel, and so it must work correctly even if the input qubit is entangled with another system.

## Further discussion

Here are a few brief, concluding remarks on teleportation.

First, teleportation is not an *application* of quantum information, it's a *protocol* for performing quantum communication.
It is therefore useful only insofar as quantum communication is useful.

Indeed, it is reasonable to speculate that teleportation could one day become a standard way to communicate quantum information, perhaps through a process known as *entanglement distillation*.
This is a process that converts a larger number of noisy (or imperfect) e-bits into a smaller number of high quality e-bits, that could then be used for noiseless or near-noiseless teleportation.
The idea is that the process of entanglement distillation is not as delicate as direct quantum communication.
We could accept losses, for instance, and if the process doesn't work out, we can just try again.
In contrast, the actual qubits we hope to communicate might be much more precious.

Finally, it should be understood that the idea behind teleportation and the way that it works is quite fundamental in quantum information and computation.
It really is a cornerstone of quantum information theory, and variations of it arise.
For example, quantum gates can be implemented through a closely related process known as *quantum gate teleportation*, which uses teleportation to apply *operations* to qubits rather than communicating them.